# MachadoGPT — modelos de linguagem estilo GPT-2 sobre a obra de Machado de Assis

Redes Neurais e Aprendizado Profundo (UnB, 2026/2) — Projeto Individual 1.
Roberto Ribeiro Corrêa De Oliveira Neto (242009936).

Pipeline completo: construção do corpus a partir da *Obra Completa* (NLTK), limpeza,
divisão por obra estratificada por gênero, tokenização e três experimentos com o
[nanoGPT](https://github.com/karpathy/nanoGPT):

| Exp. | Tokenização | Pesos iniciais | BPC (val.) |
|------|-------------|----------------|-----------:|
| E1 | caractere | aleatórios | 1,737 |
| E3 | BPE GPT-2 | aleatórios | 1,653 |
| E6 | BPE GPT-2 | GPT-2 124M | 1,586 |

Feito para o Google Colab com GPU T4 (*Ambiente de execução → Alterar tipo → T4 GPU*).
Os caminhos assumem `/content`. Detalhes e discussão no relatório (`artigo.pdf`).

**Atribuição.** Terceiros: modelo, laço de treino, carregamento dos pesos do GPT-2 e
amostragem (nanoGPT, A. Karpathy, MIT, commit `3adf61e`); pesos do GPT-2 (OpenAI, via
Hugging Face); tokenizador BPE (`tiktoken`). Autorais: download, limpeza e estatísticas
do corpus, divisão por obra, vocabulário de caracteres, configurações dos experimentos,
extração das curvas e comparação em bits por caractere.

## 1. Ambiente

In [ ]:
import nltk, pathlib, zipfile, pickle
import re, json, unicodedata, collections
import os, shutil, subprocess, math
import numpy as np
import matplotlib.pyplot as plt
import torch

assert torch.cuda.is_available(), 'sem GPU: ative uma T4 em Ambiente de execução → Alterar tipo'
!nvidia-smi

## 2. nanoGPT [Karpathy, MIT]

In [ ]:
# nanoGPT fixado num commit para reprodutibilidade.
# Clona numa pasta temporária e copia para /content/nanoGPT: funciona mesmo se a pasta
# já existir (ex.: data/machado_char gerado antes) e não apaga os dados tokenizados.
NANOGPT = '/content/nanoGPT'
NANOGPT_COMMIT = '3adf61e154c3fe3fca428ad6bc3818b27a3b8291'

if not os.path.exists(f'{NANOGPT}/train.py'):
    tmp = '/tmp/nanoGPT'
    shutil.rmtree(tmp, ignore_errors=True)
    subprocess.run(['git', 'clone', 'https://github.com/karpathy/nanoGPT.git', tmp], check=True)
    subprocess.run(['git', '-C', tmp, 'checkout', '-q', NANOGPT_COMMIT], check=True)
    shutil.copytree(tmp, NANOGPT, dirs_exist_ok=True)

!git -C {NANOGPT} log -1 --oneline
!pip install -q tiktoken

## 3. Corpus: *Obra Completa* de Machado de Assis (NLTK)

Os arquivos estão em Windows-1252; decodificar como Latin-1 apagaria o travessão (byte `0x97`).

In [ ]:
data_dir = pathlib.Path('/content/nltk_data')
nltk.download('machado', download_dir=str(data_dir))
    
corpora_dir = data_dir / 'corpora'
root = corpora_dir / 'machado'
if not root.exists():
    with zipfile.ZipFile(corpora_dir / 'machado.zip') as z:
        z.extractall(corpora_dir)

textos={}
for f in sorted(root.glob('*/*.txt')):
    if f.parent.name == 'traducao':
        continue
    textos[f"{f.parent.name}/{f.name}"] = f.read_text(encoding='cp1252')

print(len(textos), sum(len(t) for t in textos.values()))

## 4. Limpeza

Parágrafos contínuos, normalização NFC, remoção de cabeçalhos editoriais e sumários. As três traduções são excluídas.

In [ ]:
#Limpeza dos arquivos de texto, seja por corrupção, separação dos parágrafos, tirar sumários etc...

ORIGEM = pathlib.Path('/content/nltk_data/corpora/machado')
DESTINO = pathlib.Path('/content/corpus_limpo')
EXCLUIR = {'traducao'}

def paragrafos(texto):
    texto = unicodedata.normalize('NFC',texto)
    texto = texto.replace('\r\n', '\n').replace('\xa0', ' ').replace('\xad', '')
    out = []
    for bloco in re.split(r'\n\s*\n', texto):
        p = re.sub(r'\s+', ' ', bloco).strip()
        if p:
            out.append(p)
    return out

CABECALHO = re.compile(
    r'textos?[- ]fonte|texto de refer|publicad[oa]s?|nova aguilar|obra completa|'
    r'jackson|editora|biblioteca nacional|org\.|https?://|www\.|martins fontes|'
    r'sumário|\b(19[3-9]\d|20\d\d)\b', re.I)

def remover_cabecalho(ps, janela=20, max_len=200):
    ultimo = 1  # no mínimo remove a linha "Gênero, Título, Ano" e o título
    for i in range(2, min(janela, len(ps))):
        if len(ps[i]) <= max_len and CABECALHO.search(ps[i]):
            ultimo = i
    return ps[ultimo + 1:]

def chave(p):
    return re.sub(r'\W', '', p.replace('ª', 'a').replace('º', 'o')).upper()

def remover_indices(ps, max_len=120):
    out, i, removidos = [], 0, 0
    while i < len(ps):
        rotulado = chave(ps[i]) in ('ÍNDICE', 'SUMÁRIO')
        if rotulado or (i == 0 and len(ps[0]) <= max_len):
            k = i + 1 if rotulado else i
            if k < len(ps):
                alvo = chave(ps[k])
                for j in range(k + 1, len(ps)):
                    if chave(ps[j]) == alvo and j - k >= 2:
                        i, removidos = j, removidos + 1
                        break
                    if len(ps[j]) > max_len:
                        break
            if rotulado and chave(ps[i]) in ('ÍNDICE', 'SUMÁRIO'):
                i += 1 
                continue
        out.append(ps[i])
        i += 1
    return out, removidos

def limpar(caminho):
    ps = paragrafos(caminho.read_text(encoding='cp1252'))
    linha1, titulo = ps[0], ps[1]
    ano = re.findall(r'\d{3,4}', linha1)
    ps = remover_cabecalho(ps)
    ps, n_sumarios = remover_indices(ps)
    if ps and ps[-1].upper() == 'FIM':
        ps = ps[:-1]
    texto = '\n'.join(ps)
    return texto, {
        'arquivo': f'{caminho.parent.name}/{caminho.name}',
        'genero': caminho.parent.name,
        'titulo': titulo,
        'ano': int(ano[0]) if ano else None,
        'sumarios_removidos': n_sumarios,
        'caracteres': len(texto),
        'palavras': len(texto.split()),
    }

DESTINO.mkdir(parents=True, exist_ok=True)
metadados = []

for f in sorted(ORIGEM.glob('*/*.txt')):
    if f.parent.name in EXCLUIR:
        continue
    texto, meta = limpar(f)
    saida = DESTINO / meta['genero'] / f.name
    saida.parent.mkdir(exist_ok=True)
    saida.write_text(texto, encoding='utf-8')
    metadados.append(meta)

_ = (DESTINO / 'metadata.json').write_text(
    json.dumps(metadados, ensure_ascii=False, indent=2), encoding='utf-8')

## 5. Divisão treino/validação/teste

Por obra inteira, estratificada por gênero, com algoritmo guloso para aproximar 80/10/10% dos caracteres.

In [ ]:
#algoritmo para distribuir proporcionalmente os textos na proporcao correta
PROPORCOES = {'train': 0.8, 'val': 0.1, 'test': 0.1}
SPLIT_DIR = pathlib.Path('/content/dados')
metadados = json.loads((DESTINO / 'metadata.json').read_text(encoding='utf-8'))

por_genero = collections.defaultdict(list)
for m in metadados:
    por_genero[m['genero']].append(m)

for g, obras in sorted(por_genero.items()):
    print(f'{g:<12}{len(obras):>4} obras')

def dividir_genero(obras):
    obras = sorted(obras, key=lambda m: -m['caracteres'])
    total = sum(m['caracteres'] for m in obras)
    atual = {k: 0 for k in PROPORCOES}
    atribuicao = {}


    for m in obras:
        def erro_se_colocar_em(k):
            simulado = dict(atual)
            simulado[k] += m['caracteres']
            return sum(abs(simulado[j] - PROPORCOES[j] * total) for j in PROPORCOES)

        destino = min(PROPORCOES, key=erro_se_colocar_em)
        atribuicao[m['arquivo']] = destino
        atual[destino] += m['caracteres']

    return atribuicao

split = {}
for g in sorted(por_genero):
    split.update(dividir_genero(por_genero[g]))

#conferir proporções
chars = collections.defaultdict(collections.Counter)
for m in metadados:
    chars[m['genero']][split[m['arquivo']]] += m['caracteres']

print(f"{'gênero':<12}" + ''.join(f'{k:>16}' for k in PROPORCOES))  
total = collections.Counter()
for g in sorted(chars):
    c = chars[g]
    s = sum(c.values())
    total += c
    print(f'{g:<12}' + ''.join(f'{c[k]:>10,} {c[k]/s:>4.0%}' for k in PROPORCOES))

s = sum(total.values())
print(f"{'TOTAL':<12}" + ''.join(f'{total[k]:>10,} {total[k]/s:>4.0%}' for k in PROPORCOES))


# obras de validação e teste dos gêneros com poucas obras (úteis para citar no artigo)
SPLIT_DIR.mkdir(parents=True, exist_ok=True)

conjuntos = {k: [m['arquivo'] for m in metadados if split[m['arquivo']] == k]
             for k in PROPORCOES}

(SPLIT_DIR / 'split.json').write_text(
    json.dumps({'proporcoes': PROPORCOES, 'conjuntos': conjuntos},
               ensure_ascii=False, indent=2),
    encoding='utf-8')

#geração de arquivos txt para para entrada da tokenização
for k, arquivos in conjuntos.items():
    textos = [(DESTINO / arq).read_text(encoding='utf-8') for arq in arquivos]
    (SPLIT_DIR / f'{k}.txt').write_text('\n\n'.join(textos), encoding='utf-8')
    print(f"{k + '.txt':<10}{sum(len(t) for t in textos):>12,} caracteres")

## 6. Tokenização em nível de caractere [adaptada de `shakespeare_char`, nanoGPT]

Vocabulário construído só com o treino, mais um token de desconhecido.

In [ ]:
SPLIT_DIR = pathlib.Path('/content/dados')  
OUT = pathlib.Path('/content/nanoGPT/data/machado_char')
OUT.mkdir(parents=True, exist_ok=True)

textos = {k: (SPLIT_DIR / f'{k}.txt').read_text(encoding='utf-8')
          for k in ('train', 'val', 'test')}

DESCONHECIDO = '\ufffd'   # caractere �

chars = sorted(set(textos['train'])) + [DESCONHECIDO]
stoi = {ch: i for i, ch in enumerate(chars)}   # caractere -> índice
itos = {i: ch for ch, i in stoi.items()}       # índice -> caractere
vocab_size = len(chars)

print('vocab_size:', vocab_size)
print(''.join(chars))

def encode(s):
    return [stoi.get(c, stoi[DESCONHECIDO]) for c in s]

def decode(ids):
    return ''.join(itos[i] for i in ids)

amostra = 'Uma noite destas, vindo da cidade'
print(encode(amostra)[:10])
print(decode(encode(amostra)))
print('reversível:', decode(encode(amostra)) == amostra)

for k, t in textos.items():
    ids = np.array(encode(t), dtype=np.uint16)
    ids.tofile(OUT / f'{k}.bin')
    n_unk = int((ids == stoi[DESCONHECIDO]).sum())
    print(f'{k}.bin: {len(ids):,} tokens, {n_unk} desconhecidos')

with open(OUT / 'meta.pkl', 'wb') as f:
    pickle.dump({'vocab_size': vocab_size, 'itos': itos, 'stoi': stoi}, f)

m = np.memmap(OUT / 'val.bin', dtype=np.uint16, mode='r')
print(decode(m[:200].tolist()))

## 7. Experimento E1 — baseline em caractere [nanoGPT; configuração autoral]

GPT pequeno (6 camadas, 6 cabeças, 384 dim.) treinado do zero; referência para os demais.

In [ ]:
RUNS = '/content/runs'
os.makedirs(RUNS, exist_ok=True)

def ler_log(caminho):
    """Extrai (passos, loss de treino, loss de validação) do log do train.py."""
    log = open(caminho, encoding='utf-8').read()
    pts = re.findall(r'step (\d+): train loss ([\d.]+), val loss ([\d.]+)', log)
    assert pts, f'treino falhou: nenhuma avaliação no log, veja {caminho}'
    return [int(p[0]) for p in pts], [float(p[1]) for p in pts], [float(p[2]) for p in pts]

In [ ]:
%%writefile /content/nanoGPT/config/train_machado_e1.py
# E1 — baseline: nível de caractere, ~10M parâmetros
out_dir = '/content/runs/e1'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False

wandb_log = False
dataset = 'machado_char'

gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

dtype = 'float16'
compile = False

In [ ]:
%cd /content/nanoGPT
!python -u train.py config/train_machado_e1.py 2>&1 | tee /content/runs/e1.log

passos, train, val = ler_log('/content/runs/e1.log')

plt.figure(figsize=(7, 4))
plt.plot(passos, train, label='treino')
plt.plot(passos, val, label='validação')
plt.xlabel('iteração')
plt.ylabel('loss (entropia cruzada)')
plt.title('E1 — baseline em nível de caractere')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('/content/runs/e1_loss.png', dpi=200, bbox_inches='tight')
plt.show()

melhor = min(range(len(val)), key=lambda i: val[i])
print(f'melhor val loss: {val[melhor]:.4f} no passo {passos[melhor]}')

!python sample.py \
    --out_dir=/content/runs/e1 \
    --start="Uma noite destas, vindo da cidade" \
    --num_samples=3 \
    --max_new_tokens=500 \
    --temperature=0.8

## 8. Tokenização BPE do GPT-2 [adaptada do `prepare.py` de openwebtext, nanoGPT]

Cada obra termina com o token de fim de texto.

In [ ]:
import tiktoken

OUT_GPT2 = pathlib.Path('/content/nanoGPT/data/machado_gpt2')
OUT_GPT2.mkdir(parents=True, exist_ok=True)
enc = tiktoken.get_encoding('gpt2')

# exemplo de como o GPT-2 fragmenta o português
amostra = 'Uma noite destas, vindo da cidade para o Engenho Novo'
print([enc.decode([i]) for i in enc.encode_ordinary(amostra)])

estatisticas = {}
for k in ('train', 'val', 'test'):
    texto = (SPLIT_DIR / f'{k}.txt').read_text(encoding='utf-8')
    obras = texto.split('\n\n')   # as obras foram unidas com uma linha em branco na divisão (seção 5)

    ids = []
    for obra in obras:
        ids.extend(enc.encode_ordinary(obra))
        ids.append(enc.eot_token)

    arr = np.array(ids, dtype=np.uint16)
    arr.tofile(OUT_GPT2 / f'{k}.bin')
    estatisticas[k] = {'obras': len(obras), 'caracteres': len(texto), 'tokens': len(arr)}
    print(f"{k}: {len(obras)} obras, {len(texto):,} caracteres -> {len(arr):,} tokens "
          f"({len(texto) / len(arr):.2f} caracteres por token)")

(OUT_GPT2 / 'estatisticas.json').write_text(json.dumps(estatisticas, indent=2))

# verificação: decodificar o início da validação
m = np.memmap(OUT_GPT2 / 'val.bin', dtype=np.uint16, mode='r')
print(enc.decode(m[:60].tolist()))

## 9. Experimento E3 — BPE, treino do zero [nanoGPT; configuração e comparação autorais]

Muda apenas a tokenização em relação ao E1. Como a loss do E1 é por caractere e a do E3 por
token BPE, a comparação é feita em bits por caractere:
BPC = loss × (tokens / caracteres) / ln 2 (no E1, tokens = caracteres).

In [ ]:
%%writefile /content/nanoGPT/config/train_machado_e3.py
# E3 — BPE do GPT-2, treino do zero, mesma arquitetura e hiperparâmetros do E1
out_dir = '/content/runs/e3'
eval_interval = 250
eval_iters = 200
log_interval = 10
always_save_checkpoint = False

wandb_log = False
dataset = 'machado_gpt2'

gradient_accumulation_steps = 1
batch_size = 64
block_size = 256

n_layer = 6
n_head = 6
n_embd = 384
dropout = 0.2

learning_rate = 1e-3
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-4
beta2 = 0.99
warmup_iters = 100

dtype = 'float16'
compile = False

In [ ]:
%cd /content/nanoGPT
!python -u train.py config/train_machado_e3.py 2>&1 | tee /content/runs/e3.log

passos, train, val = ler_log('/content/runs/e3.log')

plt.figure(figsize=(7, 4))
plt.plot(passos, train, label='treino')
plt.plot(passos, val, label='validação')
plt.xlabel('iteração')
plt.ylabel('loss (entropia cruzada por token)')
plt.title('E3 — BPE do GPT-2, treino do zero')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('/content/runs/e3_loss.png', dpi=200, bbox_inches='tight')
plt.show()

melhor = min(range(len(val)), key=lambda i: val[i])
print(f'melhor val loss E3: {val[melhor]:.4f} no passo {passos[melhor]}')

# comparação em bits por caractere (validação)
_, _, val_e1 = ler_log('/content/runs/e1.log')
est = json.loads((OUT_GPT2 / 'estatisticas.json').read_text())
tokens_por_char = est['val']['tokens'] / est['val']['caracteres']

bpc_e1 = min(val_e1) / math.log(2)
bpc_e3 = val[melhor] * tokens_por_char / math.log(2)
print(f'E1: {bpc_e1:.3f} bits por caractere')
print(f'E3: {bpc_e3:.3f} bits por caractere')

!python sample.py \
    --out_dir=/content/runs/e3 \
    --start="Uma noite destas, vindo da cidade" \
    --num_samples=3 \
    --max_new_tokens=200 \
    --temperature=0.8

## 10. Experimento E6 — fine-tuning do GPT-2 124M [nanoGPT; pesos OpenAI; configuração autoral]

Muda apenas os pesos iniciais em relação ao E3. Na T4 leva cerca de 1h40.

In [ ]:
%%writefile /content/nanoGPT/config/train_machado_e6.py
# E6 — fine-tuning do GPT-2 (124M) pré-treinado pela OpenAI, com os mesmos dados do E3.
# Tamanho do modelo (12 camadas, 12 cabeças, 768 dim) é fixado pelos pesos do GPT-2.
out_dir = '/content/runs/e6'
eval_interval = 500
eval_iters = 200
log_interval = 10
always_save_checkpoint = False

wandb_log = False
dataset = 'machado_gpt2'
init_from = 'gpt2'

# 8 sequências por vez × 8 acumulações = 64 sequências por passo, como no E1 e no E3,
# mas com memória suficiente para caber na T4
batch_size = 8
gradient_accumulation_steps = 8
block_size = 256

dropout = 0.1

# learning rate 10x menor que no treino do zero, para não apagar o que o GPT-2 já sabe;
# mesmo número de iterações do E3 (5000), para uma comparação justa
learning_rate = 1e-4
max_iters = 5000
lr_decay_iters = 5000
min_lr = 1e-5
warmup_iters = 100

dtype = 'float16'
compile = False

In [ ]:
%cd /content/nanoGPT
!python -u train.py config/train_machado_e6.py 2>&1 | tee /content/runs/e6.log

### Comparação final em BPC

Se os logs de E1/E3 não estiverem nesta sessão, usa os valores registrados no relatório.

In [ ]:
def melhor_val(caminho, valor_anotado):
    """Melhor val loss do log; se o log não existir mais, usa o valor anotado."""
    if os.path.exists(caminho):
        return min(ler_log(caminho)[2])
    print(f'{caminho} não encontrado, usando o valor anotado: {valor_anotado}')
    return valor_anotado

est = json.loads(open('/content/nanoGPT/data/machado_gpt2/estatisticas.json').read())
tokens_por_char = est['val']['tokens'] / est['val']['caracteres']

passos, train, val = ler_log('/content/runs/e6.log')

plt.figure(figsize=(7, 4))
plt.plot(passos, train, label='treino')
plt.plot(passos, val, label='validação')
plt.xlabel('iteração')
plt.ylabel('loss (entropia cruzada por token)')
plt.title('E6 — fine-tuning do GPT-2 (124M)')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('/content/runs/e6_loss.png', dpi=200, bbox_inches='tight')
plt.show()

melhor = min(range(len(val)), key=lambda i: val[i])
print(f'melhor val loss E6: {val[melhor]:.4f} no passo {passos[melhor]}')
print(f'val loss E6 no passo 0 (GPT-2 sem fine-tuning): {val[0]:.4f}\n')

bpc = {
    'E1 (caractere, do zero)':       melhor_val('/content/runs/e1.log', 1.2038) / math.log(2),
    'E3 (BPE, do zero)':             melhor_val('/content/runs/e3.log', 2.8853) * tokens_por_char / math.log(2),
    'GPT-2 sem fine-tuning':         val[0] * tokens_por_char / math.log(2),
    'E6 (BPE, GPT-2 + fine-tuning)': val[melhor] * tokens_por_char / math.log(2),
}
for nome, v in bpc.items():
    print(f'{nome:<32}{v:.3f} bits por caractere')

%cd /content/nanoGPT
!python sample.py \
    --out_dir=/content/runs/e6 \
    --start="Uma noite destas, vindo da cidade" \
    --num_samples=3 \
    --max_new_tokens=200 \
    --temperature=0.8

## 11. Salvar resultados (opcional)

Logs, gráficos e os pesos do E6 sem o estado do otimizador.

In [ ]:
from google.colab import files

ckpt = torch.load(f'{RUNS}/e6/ckpt.pt', map_location='cpu', weights_only=False)
ckpt.pop('optimizer')
torch.save(ckpt, f'{RUNS}/e6/ckpt_pesos.pt')

!cd {RUNS} && zip -r /content/resultados.zip *.log *.png e6/ckpt_pesos.pt
files.download('/content/resultados.zip')